# Synthetic EUV / AIA — three viewpoints

Renders **synthetic SDO/AIA** maps (`I_λ = ∫ n_e² R_λ(T) dℓ`, in DN/s) for three viewpoints, using the domain-agnostic Siddon integrator `rrt.euv.run_siddon`.

- **Data**: a synthetic cube by default (so it runs anywhere). Uncomment the `load_mhd(...)` block to use your own ARMS / MAS files.
- **Rendering**: real temperature response → AIA DN/s, `sdoaia` colormap, fixed color limits, disk masked or not per view. This is why the corona shows structure instead of a saturated EM blob.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from rrt import synthdata
from rrt.io import load_mhd
from rrt.observer import make_observer, image_grid
from rrt.euv import run_siddon
from rrt.response import load_response
from rrt.viz import aia_cmap, plot_map

## 1. Choose the dataset

In [ ]:
# ---- Option A: synthetic cube (default; anisotropic so the views differ) ----
c = synthdata.make_cube('full', 'single', nr=32, nth=48, nph=64,
                        r_range=(1.0, 3.0), aniso=True)
r, th, ph, ne, T = c['r'], c['theta'], c['phi'], c['ne'], c['T']
resp_file = synthdata.write_response('/tmp/rrt_resp.npz', channels=(171,))

# ---- Option B: your own data (uncomment ONE and set the paths) ----
# ARMS wedge (density only -> isothermal):
# cube = load_mhd('data/mhd_data_0220.h5', t_iso=1.0e6)
# MAS full disk (separate rho + T, code units):
# cube = load_mhd('~/Desktop/CME/rhoC_02.h5', temperature='~/Desktop/CME/t02.h5',
#                 fmt='mas', ne_scale=1.0e8, t_scale=2.807e7, r_range=(1.03, 5.0))
# print(cube.summary())
# r, th, ph, ne, T = cube.r, cube.theta, cube.phi, cube.ne, cube.T
# resp_file = 'data/aia_temp_response_chiantifix.npz'

WAVELENGTH, RMAX, NPIX = 171, 1.5, 220
MASK_DISK = False          # True → coronagraph style (hide on-disk pixels)
VIEWS = [('Side', -70, 0), ('Tilt', 20, 20), ('Pole', -70, 89)]
T_resp, R_resp = load_response(resp_file, WAVELENGTH)

## 2. Ray-trace AIA DN/s per viewpoint

Passing the real `(T_resp, R_resp)` makes `run_siddon` return AIA DN/s directly (temperature-weighted), so cool dense plasma is suppressed. Use `FLAT_T, FLAT_R` from `rrt.response` instead if you want raw column EM.

In [ ]:
Xg, Yg, _ = image_grid(RMAX, NPIX)
images = {}
for label, phi_obs, B0 in VIEWS:
    _, e_los, x_img, y_img = make_observer(phi_obs, B0)
    images[label] = run_siddon(Xg, Yg, x_img, y_img, e_los, r, th, ph, ne, T,
                               T_resp, R_resp, Npix=NPIX, Rmax=RMAX,
                               WAVELENGTH=WAVELENGTH, plot=False, save=False)

## 3. AIA-styled three-view figure

`sdoaia` colormap, shared log color limits from the joint 50–99.9 percentile (so nothing saturates), optional disk mask + limb wireframe.

In [ ]:
# shared color limits across the three panels (avoids per-panel saturation)
allpix = np.concatenate([im[im > 0].ravel() for im in images.values()])
vlo, vhi = np.nanpercentile(allpix, [50, 99.9])

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, (label, phi_obs, B0) in zip(axes, VIEWS):
    n_obs, _, x_img, y_img = make_observer(phi_obs, B0)
    plot_map(images[label], RMAX, cmap=aia_cmap(WAVELENGTH), log=True,
             mask_disk=MASK_DISK, vlim=(vlo, vhi),
             wireframe=(n_obs, x_img, y_img),
             cbar_label=f'AIA {WAVELENGTH} Å  [DN/s]', title=label, ax=ax)
fig.suptitle(f'Synthetic AIA {WAVELENGTH} Å — three viewpoints', y=1.02)
plt.tight_layout(); plt.show()